# UM4MA266, Numerical optimization and Data science
## STEREO Project

Student : SIDATE Mathieu<br>
Sorbonne Université, 2025-2026

## But:

- on a la distance focale f et la Baseline B, on veut trouver la disparité, qui est une FONCTION d(x,y), propre à chacun des pixels; plus un objet est proche des caméras plus il sera sujet à une grande disparité.
- L'optimisation nous permet de retrouver la CARTE de disparité
- Avec ça on peut retrouver la profondeur z d'un pixel dans l'image avec la formule $d = \frac{f \cdot B}{z}$
- On a alors une carte de profondeur et on peut représenter l'image en depthscale (cf. Middleburry depthscale image stereovision)

### 1. Implementation de la méthode de block matching 

#### 1.1. Calcul du SSD (Sum of Squared differences)

On a que,
$$
C(x,y,d) = \displaystyle\sum_{(i,j)\in\cal{W}} |{I}_{L}(x+i, y+j) - {I}_{R}(x+i-d(x, y), y+j)|^2, \tag{1}
$$
avec $\cal{W}$ notre window (ou bloc) de pixels,<br><br> 
${I}_{L}$ et ${I}_{R}$ nos images gauche droite qui renvoient ici une valeur spécifique à un pixel (généralement la luminosité sur une image mis en niveau de gris).

In [29]:
import numpy as np
import cv2
import os

In [30]:
print(os.getcwd())

/Users/mathieu/academic/cs/py_stuff/opti_tp/project


In [31]:
# Méthode de SSD local avec fenêtre en brutforce

def ssd(l_img, r_img, D_max, window):
    """
    Args :
        [l_img] array de pixels (dim = résolution image) de l'image gauche ayant pour valeur la luminosité
        [r_img] // pour l'image droite
        [d] disparité (ie. écart horizontal entre 2 pixels)
        [window] taille de notre fenêtre carrée de pixel

    Outputs :
        [disp_map] carte de disparité (dim(img)) qui minimise le coût de correspondance total (+ il est faible + les images à translation d près sont similaires)
                   avec d_ij dans disp_map à valeurs dans [0,D_max]
    Assumptions :
        - dim(l_img) = dim(r_img)
        - taille de window < taille d'image
        - disparité D_max < largeur image
    """

    h, w = l_img.shape  # hauteur (height) , largeur (width) des images
    half_win = window // 2
    disp_map = np.zeros([h, w])    
    
    # création du padding
    padded_l_img = cv2.copyMakeBorder(l_img, top=half_win, bottom=half_win, left=half_win, right=half_win, borderType=cv2.BORDER_REFLECT)
    padded_r_img = cv2.copyMakeBorder(r_img, top=half_win, bottom=half_win, left=half_win, right=half_win, borderType=cv2.BORDER_REFLECT)
    
    # On parcourt tous les pixels de l'image (grâce au padding)
    for y in range(h):
        for x in range(w):
            
            best_cost = float('inf') # valeur élevée pour que tous les autres coup soient inférieurs
            
            for d in range(D_max):
                
                I_ls = padded_l_img[y:y+window, x:x+window]             # fenêtre centrée en (x,y)   
                
                if x-d >= 0 and x-d <= w: # même si le 2ème point est évident 
                    I_rs = padded_r_img[y:y+window, x-d:x-d+window]       # fenêtre centrée en (x-d, y)
                else:
                    continue
                    
                cost = np.sum((I_ls - I_rs)**2)

                if cost < best_cost:
                    disp_map[y, x] = d
                    best_cost = cost
                    
    
    return disp_map 

In [32]:
A = np.random.rand(30, 30)
B = np.random.rand(30, 30)
print(ssd(A, B, 10, 3))
ssd(A, A, 10, 3) # le coût est bien nul comme les images sont les mêmes => disparité optimale = 0

[[0. 1. 2. 3. 1. 1. 1. 0. 4. 4. 9. 4. 8. 3. 3. 8. 8. 8. 5. 4. 4. 5. 8. 5.
  8. 1. 1. 1. 1. 1.]
 [0. 1. 2. 2. 2. 1. 1. 0. 4. 4. 4. 4. 8. 8. 8. 8. 8. 8. 2. 2. 4. 6. 8. 8.
  8. 1. 1. 1. 1. 1.]
 [0. 1. 2. 3. 3. 3. 6. 7. 4. 9. 3. 3. 0. 8. 8. 8. 8. 8. 0. 0. 0. 8. 4. 4.
  6. 6. 6. 0. 5. 0.]
 [0. 0. 2. 3. 3. 3. 0. 0. 5. 3. 7. 9. 9. 0. 9. 5. 8. 8. 0. 1. 0. 6. 4. 4.
  1. 8. 8. 8. 5. 0.]
 [0. 0. 0. 0. 1. 1. 0. 0. 7. 7. 7. 9. 7. 0. 8. 9. 8. 8. 5. 1. 0. 0. 0. 6.
  4. 2. 5. 7. 5. 6.]
 [0. 0. 0. 0. 0. 0. 0. 0. 4. 4. 2. 9. 7. 7. 8. 9. 9. 9. 9. 4. 3. 3. 0. 0.
  4. 2. 5. 7. 7. 1.]
 [0. 0. 0. 1. 0. 0. 0. 0. 0. 6. 2. 9. 9. 0. 8. 3. 3. 2. 9. 4. 0. 3. 3. 3.
  9. 9. 9. 7. 3. 3.]
 [0. 1. 2. 0. 4. 5. 5. 0. 4. 6. 6. 6. 9. 9. 9. 3. 3. 2. 9. 1. 4. 4. 4. 4.
  9. 9. 9. 9. 2. 3.]
 [0. 1. 2. 2. 1. 5. 5. 5. 4. 6. 6. 9. 9. 3. 3. 8. 8. 2. 9. 2. 9. 3. 9. 9.
  9. 9. 0. 0. 2. 3.]
 [0. 1. 2. 2. 0. 5. 1. 7. 4. 7. 8. 8. 4. 9. 9. 3. 8. 8. 9. 2. 2. 2. 9. 2.
  0. 0. 0. 0. 0. 2.]
 [0. 1. 1. 1. 4. 4. 6. 7. 8. 5. 8. 8. 8. 0. 9. 0. 

array([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0

Ca a l'air de marcher en ayant pris l'exemple d'images aléatoire.

### 2. Implémentation de la méthode semi-globale (SGM) 

#### 2.1. Calcul de l'énergie globale

### Formulation de l’énergie

$$
E(d) = \sum_{x,y} C(x, y, d(x,y)) + \lambda \sum_{(p,q) \in \mathcal{N}} V(d(p) - d(q))
$$


### 1. Data term

$$
C(x, y, d)
$$

  * SSD (Sum of Squared Differences)

+ ce terme est faible + les images sont similaires 


### 2. Terme de régularisation

$$
V(d(p) - d(q))
$$

* V est la fonction du SGM introduit en 2005 (cf. wikipédia SGM) 
$$
V(d_p, d_q) =
\begin{cases}
0 & \text{if } d_p = d_q, \\
P_1 & \text{if } |d_p - d_q| = 1, \\
P_2 & \text{if } |d_p - d_q| > 1.
\end{cases}
$$

In [33]:
# correspond au ssd précédent

def compute_cost_volume(l_img, r_img, D_max, window):
    """
    Args :
        [l_img] array de pixels (dim = résolution image) de l'image gauche ayant pour valeur la luminosité
        [r_img] // pour l'image droite
        [D_max] disparité (ie. écart horizontal entre 2 pixels) maximale autorisée
        [window] taille de notre fenêtre carrée de pixel

    Outputs :
        [cost] coût de correspondance total pour chaque disparité (+ il est faible + les images à translation d près sont similaires)
        
    Assumptions :
        - dim(l_img) = dim(r_img)
        - taille de window < taille d'image
        - disparité D_max < largeur image    
    """
    
    h, w = l_img.shape
    cost = np.zeros((h, w, D_max), dtype=np.float32)
    half_win = window // 2
    padded_l_img = cv2.copyMakeBorder(l_img, top=half_win, bottom=half_win, left=half_win, right=half_win, borderType=cv2.BORDER_REFLECT)
    padded_r_img = cv2.copyMakeBorder(r_img, top=half_win, bottom=half_win, left=half_win, right=half_win, borderType=cv2.BORDER_REFLECT)

    # On parcourt tous les pixels de l'image (grâce au padding)
    for y in range(h):
        for x in range(w):
            
            for d in range(D_max):
                
                I_ls = padded_l_img[y:y+window, x:x+window]             # fenêtre centrée en (x,y)   
                
                if x-d >= 0 and x-d <= w: # même si le 2ème point est évident 
                    I_rs = padded_r_img[y:y+window, x-d:x-d+window]       # fenêtre centrée en (x-d, y)
                else:
                    continue
                    
                cost[:, :, d] = np.sum((I_ls - I_rs)**2)
                
    return cost

In [34]:
# correspond à la partie régularisation

def sgm_aggregate(cost, P1=5, P2=100, lambd=1):
    """
    Args : 
        [cost] coût de correspondance total pour chaque disparité
        [P1] pénalité pour un petit écart de disparité
        [P2] pénalité pour un grand écart de disparité

    Outputs :
        [S] coût cumulé de tous les voisins
    """
    H, W, D = cost.shape
    directions = [
        (0, 1),   # est
        (0, -1),  # ouest
        (1, 0),   # sud
        (-1, 0),  # nord
        (1, 1),   # sud est
        (-1, -1), # nord ouest
        (1, -1),  # sud ouest
        (-1, 1),  # nord est
    ]

    S = np.zeros_like(cost)

    for dy, dx in directions:
        L = np.zeros_like(cost)

        y_range = range(H) if dy >= 0 else range(H-1, -1, -1)
        x_range = range(W) if dx >= 0 else range(W-1, -1, -1)

        for y in y_range:
            for x in x_range:
                y_prev = y - dy
                x_prev = x - dx

                if 0 <= y_prev < H and 0 <= x_prev < W:
                    prev = L[y_prev, x_prev]

                    prev_min = prev.min()

                    l1 = prev
                    l2 = np.roll(prev, 1) + lambd * P1
                    l3 = np.roll(prev, -1) + lambd * P1
                    l4 = np.full(D, prev_min + lambd * P2, dtype=np.float32)

                    l2[0] = 1e9
                    l3[-1] = 1e9

                    L[y, x] = cost[y, x] + np.minimum.reduce([l1, l2, l3, l4]) - prev_min
                else:
                    L[y, x] = cost[y, x]

        S += L

    return S

In [35]:
def sgm_disparity_optimized(imgL, imgR, D_max=64, window=15, P1=10, P2=100, lambd=1):
    imgL = imgL.astype(np.float32)
    imgR = imgR.astype(np.float32)
    
    cost = compute_cost_volume(imgL, imgR, D_max, window)
    S = sgm_aggregate(cost, P1, P2, lambd)
    disp = np.argmin(S, axis=2)

    return disp.astype(np.float32)

In [36]:
A = np.random.rand(70, 70)
B = np.random.rand(70, 70)
sgm_disparity_optimized(A, B)

array([[42., 42., 42., ..., 42., 42., 42.],
       [42., 42., 42., ..., 42., 42., 42.],
       [42., 42., 42., ..., 42., 42., 42.],
       ...,
       [42., 42., 42., ..., 42., 42., 42.],
       [42., 42., 42., ..., 42., 42., 42.],
       [42., 42., 42., ..., 42., 42., 42.]], shape=(70, 70), dtype=float32)

normal car bruit gaussien donc ça sort une constante de disp pour tous les pixels

In [37]:
sgm_disparity_optimized(A, A)

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]], shape=(70, 70), dtype=float32)

normal car même image

## 3. Influence des paramètres en stéréovision

### Taille de la fenêtre

**Effets :**

* Petite fenêtre :

  * plus précis localement
  * plus sensible au bruit
    
* Grande fenêtre :

  * moins de bruit
  * moins de détails

* petite fenêtre meilleure pour endroits avec détails
* grande fenêtre meilleure pour endroits uniformes

### Poids de régularisation 

**Effets :**

* lambda faible :

  *  fidèle aux données
  *  carte bruitée

* lambda élevé :

  *  carte lissé et cohérente
  *  perte des discontinuités

* avec des scènes simples, on augmente lambda
* avec des scènes avec objets complexes, on réduit lambda

### Pénalisations

**Effets :**

* P1 (petite pénalité)

  * plus P1 petit + on autorise les détails fins
  * plus P1 grand + on rend la carte lisse

* P2 (grosse pénalité)

  * plus P2 petit + on autorise les discontinuités
  * plus P2 grand + on crée des surfaces lisses

On a,
$$
P_2 \gg P_1
$$


## 4. Régularisation du problème

Notre but est de trouver $d^* \in \mathbb{N}^{h * w}$ tel que $d^* = \displaystyle \arg\min_{d \in \mathbb{N}^{h * w}} C_{tot}(d)$

<span style="margin-left:2em;"> où, $C_{tot}(d) = \displaystyle \sum_{(x,y) \in I} C(x,y,d(x, y))$</span>
<span style="margin-left:2em;"> avec $I$ la dimension de la matrice de pixels. </span>

On aimerait utiliser les techniques d'optimisation usuelles sur $C_{tot}$ pour trouver un minimiseur mais un problème premier étant que la fonction n'est pas régulière.

En effet, d(x, y) est à valeur dans $\mathbb{N}$ et on ne sait rien de la différentiabilité de la fonction $I_{R}$ qui est cruciale dans le calcul de différentielle de $(1)$.

Prenons le cas simple du SSD local sans fenêtre :
$$
C(x,y,d) = |I_{L}(x,y) - I_{R}(x-d(x,y),y)|^2
$$

### 4.1. Prolongement continue de C

On construit $\bar{C}$ tel que $\forall d \in \mathbb{R_{+}}, \bar{C}(x,y,d)$ différentiable et $\bar{C}(x,y,d) = C(x,y,d), \forall d \in \mathbb{N} $

On va utiliser la technique d'interpolation linéaire pour se retrouver avec une fonction qui est C1 par morceaux (les points compliqués étant les points entiers qui sont les démarcations de notre maillage) mais ce n'est pas un problème car les techniques d'optimisations ne tomberont presque sûrement pas en ces points.

#### Interpolation linéaire

$\large \bar{f}(x) = \frac{x_{b} - x}{x_{b} - x_{a}}y_{a} + \frac{x - x_{a}}{x_{b} - x_{a}}y_{b},  \forall x \in [x_{a},x_{b}]$</br></br>
tel que $f(x_a)=y_a$ et $f(x_b)=y_b$

Ce qui pose problème c'est quand $u = (x-d) \in \mathbb{R}_{+}$ </br>
donc on pose $\forall u \in \mathbb{R_{+}}, \bar{I}(u,y) = (\lceil u \rceil - u)I(\lfloor u \rfloor,y) + (u - \lfloor u \rfloor)I(\lceil u \rceil,y)$

On a donc, 
$$
\bar{C}(x,y,d) = |I_{L}(x,y) - \bar{I}_{R}(x-d,y)|^2
$$
et,
$$
\bar{C}_{tot}(d) = \displaystyle \sum_{(x,y)} |I_{L}(x,y) - \bar{I}_{R}(x-d,y)|^2
$$
$$
\frac{\partial{\bar{C}_{tot}}}{\partial{d}} = 2 \displaystyle \sum_{(x,y)} (I_R(i+1,y) - I_R(i,y))(I_L(x,y) - \bar{I}_R(x-d,y))
$$
<span style="margin-left:2em;">où, $i = \lfloor u \rfloor$ </span>

### 4.2. Optimisation du problème

Je choisis une méthode de descente de gradient à pas fixe.

$$
d^{(k+1)} = d^{(k)} - \rho \nabla \bar{C}_{tot}(d^{(k)})
$$

## 5. Pyramide multi-échelle (extension du projet)

## Strategie:

- downscale de 2^K les images
- brutforce disparity d_prov
- upscale x2 et faire descente de gradient sur un intervalle [d_prov +- delta] avec delta de l'ordre de (2-4 px)
- boucle où on refait la même chose ie. upscale et descente de gradient avec un intervalle centré en le nv d_prov obtenu avant (tjrs delta = 2-4px)
- quand on la fait jusqu'à la Kème fois, ie. pleine résolution, on retourne la disparité

In [38]:
def build_pyramid(img, num_levels):
    pyramid = [img]
    for _ in range(num_levels - 1):
        pyramid.append(cv2.pyrDown(pyramid[-1]))
    return pyramid

In [39]:
def upsample_disparity(disp, target_shape):
    H, W = target_shape
    disp_up = cv2.resize(disp, (W, H), interpolation=cv2.INTER_LINEAR)
    return disp_up * 2.0

In [40]:
def compute_cost_volume_local(img_left, img_right, disp_init, delta, window_size):

    H, W = img_left.shape
    half_w = window_size // 2

    padded_L = cv2.copyMakeBorder(img_left, half_w, half_w, half_w, half_w, cv2.BORDER_REFLECT)
    padded_R = cv2.copyMakeBorder(img_right, half_w, half_w, half_w, half_w, cv2.BORDER_REFLECT)

    disparities = np.arange(-delta, delta + 1)
    D = len(disparities)

    cost_volume = np.zeros((H, W, D), dtype=np.float32)

    for k, d in enumerate(disparities):
        shifted_R = np.roll(padded_R, -d, axis=1)

        diff = (padded_L - shifted_R) ** 2

        cost = cv2.boxFilter(diff, ddepth=-1, ksize=(window_size, window_size))

        cost_volume[:, :, k] = cost[half_w:-half_w, half_w:-half_w]

    return cost_volume, disparities

def refine_disparity_local(img_left, img_right, disp_init,
                           delta=2, window_size=7):

    cost_volume, disparities = compute_cost_volume_local(
        img_left, img_right, disp_init, delta, window_size
    )

    best_idx = np.argmin(cost_volume, axis=2)

    disp_refined = disp_init + disparities[best_idx]

    return disp_refined

def multiscale_disparity(img_left, img_right,
                              num_levels=4,
                              delta=2,
                              window_size=7):

    pyr_L = build_pyramid(img_left, num_levels)
    pyr_R = build_pyramid(img_right, num_levels)

    disp = None

    for level in range(num_levels - 1, -1, -1):

        IL = pyr_L[level]
        IR = pyr_R[level]

        if disp is None:
            
            num_disp = 64 // (2**level)
            num_disp = max(16, num_disp // 16 * 16)

            stereo = cv2.StereoBM_create(
                numDisparities=num_disp,
                blockSize=15
            )

            disp = stereo.compute(IL, IR).astype(np.float32) / 16.0

        else:
            disp = upsample_disparity(disp, IL.shape)

            disp = refine_disparity_local(
                IL, IR, disp,
                delta=delta,
                window_size=window_size
            )

    return disp

def stereo_multiechelle(img_left, img_right):

    disp = multiscale_disparity(
        img_left,
        img_right,
        num_levels=4,
        delta=2,        
        window_size=7     
    )

    return disp

In [42]:
imgL = cv2.imread("img_files/couch/im0.png", cv2.IMREAD_GRAYSCALE)
imgR = cv2.imread("img_files/couch/im1.png", cv2.IMREAD_GRAYSCALE)
stereo_multiechelle(imgL, imgR)

array([[-10.      , -10.      , -14.      , ..., -11.      ,  -9.      ,
        -14.      ],
       [-11.      ,  -9.984375, -13.953125, ..., -11.      ,  -9.      ,
        -10.      ],
       [-10.      ,  -9.953125, -13.859375, ...,  -9.      ,  -9.      ,
        -11.      ],
       ...,
       [-10.      ,  -8.75    , -11.25    , ...,  -8.625   ,  -8.875   ,
        -10.      ],
       [-10.      ,  -9.75    , -11.25    , ..., -10.875   ,  -9.625   ,
        -10.      ],
       [-10.      ,  -8.75    , -11.25    , ..., -11.      , -10.      ,
        -10.      ]], shape=(1992, 2300))